# Week 7 — Feature Engineering

## Better Inputs, Better Models

In this task, I will study how feature engineering can improve a machine learning model.

I will train the same K-Nearest Neighbors model twice:

1. Using the original raw features
2. Using encoded, scaled and newly created features

The purpose is to check whether improving the inputs can improve model accuracy.

## Project Question

**Can feature engineering improve KNN accuracy when predicting the wine class from chemical measurements?**

The target is the wine class, and the input features are different chemical measurements from each wine sample.

## What Is Feature Engineering?

Feature engineering means creating, transforming or selecting the input columns used by a machine learning model.

A model learns from the information provided through its features. If the features are unclear, poorly scaled or not useful, changing the algorithm may not solve the problem.

Improving the features can sometimes produce a bigger improvement than changing to a more advanced model.

## Feature Scaling

Numerical columns can have very different value ranges.

K-Nearest Neighbors calculates the distance between data points. A feature containing large values may control the distance calculation more than a feature containing small values.

StandardScaler places the numerical features on a more comparable scale. This helps KNN consider the features more fairly.

## Encoding Categorical Data

Machine learning models normally require numerical inputs and cannot directly understand text categories.

One-hot encoding converts each category into a separate column containing 0 or 1.

For example, the categories Low, Medium and High can become:

- alcohol_level_Low
- alcohol_level_Medium
- alcohol_level_High

## Creating New Features

A new feature can be created by combining information from existing columns.

A useful new feature may show a relationship that is not as clear when the original columns are viewed separately.

In this task, I will create a ratio using total phenols and flavanoids.

## Feature Selection

More features do not always produce a better model.

Some features may contain repeated information, noise or information that is not useful for the prediction.

Feature selection means keeping the features that help the model and avoiding unnecessary features.

In [1]:
# Import the required libraries

import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [2]:
# Load the Wine dataset

wine = load_wine()

print("The Wine dataset has been loaded successfully.")

The Wine dataset has been loaded successfully.


In [3]:
# Convert the dataset into a pandas DataFrame

df = pd.DataFrame(
    wine.data,
    columns=wine.feature_names
)

# Add the target column

df["target"] = wine.target

# Add readable target names

df["wine_class"] = df["target"].map({
    0: "Class 0",
    1: "Class 1",
    2: "Class 2"
})

df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target,wine_class
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0,Class 0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0,Class 0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0,Class 0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0,Class 0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0,Class 0


## Dataset Information

Each row represents one wine sample.

The input columns contain chemical measurements such as:

- Alcohol
- Malic acid
- Magnesium
- Total phenols
- Flavanoids
- Proline

The target column contains the correct wine class.

In [4]:
# Check the size of the dataset

print("Dataset shape:", df.shape)

# Check the number of samples in each class

print("\nSamples in each wine class:")
print(df["wine_class"].value_counts())

Dataset shape: (178, 15)

Samples in each wine class:
wine_class
Class 1    71
Class 0    59
Class 2    48
Name: count, dtype: int64


In [5]:
# Display information about the dataset

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       178 non-null    float64
 1   malic_acid                    178 non-null    float64
 2   ash                           178 non-null    float64
 3   alcalinity_of_ash             178 non-null    float64
 4   magnesium                     178 non-null    float64
 5   total_phenols                 178 non-null    float64
 6   flavanoids                    178 non-null    float64
 7   nonflavanoid_phenols          178 non-null    float64
 8   proanthocyanins               178 non-null    float64
 9   color_intensity               178 non-null    float64
 10  hue                           178 non-null    float64
 11  od280/od315_of_diluted_wines  178 non-null    float64
 12  proline                       178 non-null    float64
 13  targe

In [6]:
# Display the statistical summary

df.describe()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
count,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000
mean,13.000618,2.336348,2.366517,19.494944,99.741573,2.295112,2.029270,0.361854,1.590899,5.058090,0.957449,2.611685,746.893258,0.938202
std,0.811827,1.117146,0.274344,3.339564,14.282484,0.625851,0.998859,0.124453,0.572359,2.318286,0.228572,0.709990,314.907474,0.775035
min,11.030000,0.740000,1.360000,10.600000,70.000000,0.980000,0.340000,0.130000,0.410000,1.280000,0.480000,1.270000,278.000000,0.000000
25%,12.362500,1.602500,2.210000,17.200000,88.000000,1.742500,1.205000,0.270000,1.250000,3.220000,0.782500,1.937500,500.500000,0.000000
50%,13.050000,1.865000,2.360000,19.500000,98.000000,2.355000,2.135000,0.340000,1.555000,4.690000,0.965000,2.780000,673.500000,1.000000
75%,13.677500,3.082500,2.557500,21.500000,107.000000,2.800000,2.875000,0.437500,1.950000,6.200000,1.120000,3.170000,985.000000,2.000000
max,14.830000,5.800000,3.230000,30.000000,162.000000,3.880000,5.080000,0.660000,3.580000,13.000000,1.710000,4.000000,1680.000000,2.000000


In [7]:
# Check for missing values

print("Total missing values:", df.isnull().sum().sum())

# Check for duplicate rows

print("Duplicate rows:", df.duplicated().sum())

Total missing values: 0
Duplicate rows: 0


## Initial Data Observation

The Wine dataset contains numerical chemical measurements and three target classes.

The first inspection shows that the dataset has no missing values. The numerical features have very different value ranges. For example, some features contain small decimal values while features such as proline contain much larger values.

This difference in scale may affect KNN because KNN uses distance when making predictions.

In [8]:
# Create a categorical feature from the alcohol column

df["alcohol_level"] = pd.cut(
    df["alcohol"],
    bins=[-np.inf, 12.5, 13.5, np.inf],
    labels=["Low", "Medium", "High"]
)

df[["alcohol", "alcohol_level"]].head(10)

,alcohol,alcohol_level
0,14.23,High
1,13.20,Medium
2,13.16,Medium
3,14.37,High
4,13.24,Medium
5,14.20,High
6,14.39,High
7,14.06,High
8,14.83,High
9,13.86,High


## Reason for Creating Alcohol Level

The original alcohol column contains continuous numerical values.

I grouped these values into Low, Medium and High categories to create a simpler categorical description of the alcohol measurement.

This allows me to practise converting a text category into numbers using one-hot encoding.

In [9]:
# Check the number of samples in each alcohol category

print(df["alcohol_level"].value_counts())

alcohol_level
Medium    66
Low       57
High      55
Name: count, dtype: int64


In [10]:
# Create a new feature from existing columns

df["phenol_ratio"] = (
    df["total_phenols"] /
    (df["flavanoids"] + 0.001)
)

df[
    [
        "total_phenols",
        "flavanoids",
        "phenol_ratio"
    ]
].head()

,total_phenols,flavanoids,phenol_ratio
0,2.80,3.06,0.914734
1,2.65,2.76,0.959797
2,2.80,3.24,0.863931
3,3.85,3.49,1.102836
4,2.80,2.69,1.040505


## Reason for Creating Phenol Ratio

I created the phenol_ratio feature by dividing total phenols by flavanoids.

Both columns describe chemical properties of the wine. The ratio may help the model understand the relationship between these two measurements instead of looking at each measurement only by itself.

This does not guarantee better accuracy, so the result must be tested honestly.

In [11]:
# Store the names of the original numerical features

raw_feature_columns = list(wine.feature_names)

# Create the raw input features and target

X_raw = df[raw_feature_columns]
y = df["target"]

print("Raw feature shape:", X_raw.shape)
print("Target shape:", y.shape)

Raw feature shape: (178, 13)
Target shape: (178,)


## Raw Features

The raw model will use the 13 original numerical features.

The raw numerical values will not be scaled, and the newly created categorical and ratio features will not be included.

This will provide the starting accuracy for comparison.

In [12]:
# Create one train/test split using the row indexes

train_indexes, test_indexes = train_test_split(
    df.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(train_indexes))
print("Testing samples:", len(test_indexes))

Training samples: 142
Testing samples: 36


## Fair Comparison

The row indexes were split only once.

The same training rows and testing rows will be used for both the raw model and the engineered model.

This makes the comparison fair because both versions are evaluated using exactly the same test examples.

In [13]:
# Prepare the raw training and testing data

X_raw_train = X_raw.loc[train_indexes]
X_raw_test = X_raw.loc[test_indexes]

y_train = y.loc[train_indexes]
y_test = y.loc[test_indexes]

print("Raw training shape:", X_raw_train.shape)
print("Raw testing shape:", X_raw_test.shape)

Raw training shape: (142, 13)
Raw testing shape: (36, 13)


In [14]:
# Create the raw KNN model

raw_model = KNeighborsClassifier(
    n_neighbors=5
)

# Train the raw model

raw_model.fit(
    X_raw_train,
    y_train
)

# Make raw predictions

raw_predictions = raw_model.predict(
    X_raw_test
)

# Calculate raw accuracy

raw_accuracy = accuracy_score(
    y_test,
    raw_predictions
)

print(f"Raw Model Accuracy: {raw_accuracy:.2%}")

Raw Model Accuracy: 80.56%


## Raw Model Observation

The raw KNN model used the original numerical values without feature scaling.

Some Wine dataset features have much larger values than others. Because KNN uses distance, the large-value features may have influenced the prediction more strongly.

In [15]:
# Select the features for the engineered version

engineered_data = df[
    raw_feature_columns +
    [
        "alcohol_level",
        "phenol_ratio"
    ]
].copy()

# Apply one-hot encoding to alcohol_level

engineered_data = pd.get_dummies(
    engineered_data,
    columns=["alcohol_level"],
    prefix="alcohol_level",
    dtype=int
)

engineered_data.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,phenol_ratio,alcohol_level_Low,alcohol_level_Medium,alcohol_level_High
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0.914734,0,0,1
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0.959797,0,1,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0.863931,0,1,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,1.102836,0,0,1
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,1.040505,0,1,0


In [16]:
# Display the encoded category columns

encoded_columns = [
    column
    for column in engineered_data.columns
    if "alcohol_level_" in column
]

print("Encoded categorical columns:")
print(encoded_columns)

Encoded categorical columns:
['alcohol_level_Low', 'alcohol_level_Medium', 'alcohol_level_High']


## Encoding Observation

The alcohol_level text categories were converted into separate numerical columns.

A value of 1 means that a wine belongs to that category, while a value of 0 means that it does not belong to that category.

The KNN model can now use this information because the categories are represented numerically.

In [17]:
# Use the same training and testing rows

X_engineered_train = engineered_data.loc[
    train_indexes
].copy()

X_engineered_test = engineered_data.loc[
    test_indexes
].copy()

print(
    "Engineered training shape:",
    X_engineered_train.shape
)

print(
    "Engineered testing shape:",
    X_engineered_test.shape
)

Engineered training shape: (142, 17)
Engineered testing shape: (36, 17)


In [18]:
# Identify the numerical columns that need scaling

numeric_columns = (
    raw_feature_columns +
    ["phenol_ratio"]
)

# Create the scaler

scaler = StandardScaler()

# Learn the scaling values from the training data

X_engineered_train[numeric_columns] = (
    scaler.fit_transform(
        X_engineered_train[numeric_columns]
    )
)

# Apply the same scaling to the testing data

X_engineered_test[numeric_columns] = (
    scaler.transform(
        X_engineered_test[numeric_columns]
    )
)

X_engineered_train.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,phenol_ratio,alcohol_level_Low,alcohol_level_Medium,alcohol_level_High
36,0.385801,-0.637871,1.776668,-1.224532,0.696430,0.526865,0.732292,-0.169549,-0.415783,-0.167467,0.624378,0.252908,0.467725,-0.628299,0,1,0
30,0.948519,-0.765445,1.253174,0.853284,0.091785,1.172795,1.333181,-0.590457,1.349742,0.305303,1.067155,0.151048,1.815768,-0.693403,0,0,1
26,0.523354,-0.519409,0.954034,-1.046433,-0.445678,0.930572,1.006382,-0.169549,-0.260002,-0.081509,-0.128343,0.893172,1.516203,-0.629310,0,1,0
12,0.973529,-0.555859,0.168793,-1.076116,-0.714409,0.526865,0.816627,-0.590457,0.363125,0.262324,0.890044,0.427526,1.932265,-0.667227,0,0,1
148,0.435820,0.820120,0.056615,0.556453,-0.512860,-0.555068,-1.291756,0.756449,-0.606183,1.474335,-1.766619,-1.435059,-0.297831,1.541655,0,1,0


## Scaling Observation

StandardScaler learned the scaling information only from the training data.

The same transformation was then applied to the test data.

This avoids using information from the test set while preparing the model and keeps the evaluation honest.

In [19]:
# Create the engineered KNN model

engineered_model = KNeighborsClassifier(
    n_neighbors=5
)

# Train the engineered model

engineered_model.fit(
    X_engineered_train,
    y_train
)

# Make predictions

engineered_predictions = engineered_model.predict(
    X_engineered_test
)

# Calculate accuracy

engineered_accuracy = accuracy_score(
    y_test,
    engineered_predictions
)

print(
    f"Engineered Model Accuracy: "
    f"{engineered_accuracy:.2%}"
)

Engineered Model Accuracy: 100.00%


In [20]:
# Create a comparison table

comparison_results = pd.DataFrame({
    "Feature Version": [
        "Raw Features",
        "Engineered Features"
    ],
    "Accuracy": [
        raw_accuracy,
        engineered_accuracy
    ]
})

comparison_results["Accuracy (%)"] = (
    comparison_results["Accuracy"] * 100
).round(2)

comparison_results[
    [
        "Feature Version",
        "Accuracy (%)"
    ]
]

,Feature Version,Accuracy (%)
0,Raw Features,80.56
1,Engineered Features,100.00


In [21]:
# Calculate the accuracy difference

accuracy_difference = (
    engineered_accuracy -
    raw_accuracy
) * 100

print(
    f"Accuracy difference: "
    f"{accuracy_difference:.2f} percentage points"
)

if accuracy_difference > 0:
    print("Feature engineering improved the model.")
elif accuracy_difference < 0:
    print("Feature engineering reduced the model accuracy.")
else:
    print("Feature engineering did not change the accuracy.")

Accuracy difference: 19.44 percentage points
Feature engineering improved the model.


## Did Feature Engineering Help?

Yes, feature engineering helped in this comparison.

The raw KNN model achieved approximately 80.56% accuracy, while the engineered model achieved approximately 100.00% accuracy.

The engineered model therefore improved by approximately 19.44 percentage points.

I think the biggest reason was feature scaling. The original Wine dataset contains columns with very different value ranges, and KNN calculates distance between samples. StandardScaler allowed the numerical features to contribute more fairly.

One-hot encoding also allowed the model to use the alcohol_level category, while the phenol_ratio feature gave it additional information about the relationship between total phenols and flavanoids.

However, this comparison does not prove exactly how much each individual change helped. I would need to test scaling, encoding and the new feature separately to measure their individual effects.

## Final Conclusion

In this task, I learned that improving the input features can make a large difference to machine learning performance.

I encoded a categorical feature, scaled the numerical columns and created a new ratio feature. I then trained the same KNN model using both the raw features and the engineered features.

The engineered version performed better on the same test set. This showed me that model performance depends not only on the algorithm but also on how clearly the data is prepared and represented.